In [1]:
translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']
joint_names = [
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip",
    "index_mcp_roll",
    "index_mcp_pitch",
    "index_pip",
    "index_dip",
    "middle_mcp_roll",
    "middle_mcp_pitch",
    "middle_pip",
    "middle_dip",
    "ring_mcp_pitch",
    "ring_pip",
    "ring_dip",
    "pinky_mcp_pitch",
    "pinky_pip",
    "pinky_dip"
]

thumb = {
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip"
}

In [2]:
import numpy as np
from collections import defaultdict
import scipy.spatial.transform as transform
import numpy as np
from scipy.spatial.transform import Rotation as R

grasp_poses =np.load('/home/guizhewei/guizhewei/grasp_pose_dataset/unoptimized/grasp_poses_1021_0045.npy', allow_pickle=True).item()
# grasp_poses =np.load('/home/ubuntu/Documents/DexYCB/grasp_poses_opt.npy', allow_pickle=True).item()
grasp_poses.keys()
# obj_idx = 3

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49])

In [70]:
# grasp_poses[obj_idx].keys()


In [71]:
# grasp_poses[obj_idx]['target_object_name']

In [72]:
# grasp_poses[obj_idx]['robot_pose'][0]

In [3]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def euler_to_rotation_matrix(euler_angles):
    rotation = R.from_euler('XYZ', euler_angles, degrees=False)
    return rotation.as_matrix()


def quaternion_to_rotation_matrix(quaternion):
    rotation = R.from_quat(quaternion)
    return rotation.as_matrix()


def object_pose_to_matrix(position, quaternion):
    """
    Converts object pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    quaternion = np.concatenate([quaternion[1:4], quaternion[0:1]]) # wxyz-> xyzw
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix


def hand_pose_to_matrix(position, quaternion):
    """
    Converts object pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix

In [4]:
def get_obj_centric_pose_for_opt(grasp_poses, obj_idx):
    ''' Obj pose '''
    obj_pos = grasp_poses[obj_idx]['target_pose_world'][0].p
    obj_quat = grasp_poses[obj_idx]['target_pose_world'][0].q
    object_pose = object_pose_to_matrix(obj_pos, obj_quat)
    # print(f"Object Position: {obj_pos}, Object Quaternion: {obj_quat}")
    # print(f"Object Pose: {object_pose}")

    ''' Hand pose '''
    hand_pos = grasp_poses[obj_idx]['robot_pose'][0][:3]
    hand_euler = grasp_poses[obj_idx]['robot_pose'][0][3:6]
    hand_quat = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_quat()
    hand_6drot = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_matrix()
    hand_6drot = hand_6drot[:, :2].T.ravel().tolist()
    # print(f"Hand Position: {hand_pos}, Hand Quaternion: {hand_quat}")
    # print(f"Hand 6drot: {hand_6drot}")
    # print(grasp_poses[obj_idx]['robot_pose'][0])

    ''' object-centric '''
    W_T_O = object_pose
    W_T_H = hand_pose_to_matrix(hand_pos, hand_quat)

    O_T_H = np.linalg.inv(W_T_O) @ W_T_H
    print(O_T_H)
    t_oh  = O_T_H[:3, 3]
    R_oh  = O_T_H[:3, :3]

    euler_oh = R.from_matrix(R_oh).as_euler('XYZ', degrees=False)
    print(f"Object-Centric Hand Euler (XYZ, rad): {euler_oh}")
    print(f"Object-Centric Hand Position: {t_oh}")

    hand_6drot = R_oh
    hand_6drot = hand_6drot[:, :2].T.ravel().tolist()
    hand_pos = t_oh
    hand_euler = euler_oh

    return hand_pos, hand_euler, object_pose, hand_6drot

In [5]:
grasp_poses

{0: {'target_object_name': '024_bowl',
  'target_object_idx': 13,
  'target_pose_camera': [array([-0.5440993 , -0.3518824 ,  0.58101845,  0.49249598,  0.24959724,
          -0.14965104,  0.90708315], dtype=float32)],
  'target_pose_world': [Pose([0.506014, 0.254884, 0.18723], [-0.616193, -0.552497, -0.517738, -0.216795])],
  'camera_pose': Pose([1.01468, 0.777723, 0.799924], [0.0533614, -0.230272, -0.91078, 0.338536]),
  'robot_names': [<RobotName.omni: 8>],
  'robot_pose': [array([ 4.22882169e-01,  3.90678406e-01,  1.82086930e-01,  1.84016883e+00,
           2.50069976e-01,  7.12601766e-02, -1.90658525e-01,  2.62600005e-01,
           1.05808645e-01,  3.96602511e-01,  5.98770201e-01,  1.94344074e-01,
          -1.00000005e-03, -1.00000005e-03,  3.47661761e-01,  5.80507710e-01,
           1.07427463e-01, -1.00000005e-03, -1.00000005e-03,  3.47661761e-01,
           5.80507710e-01,  3.35082188e-02, -1.11620005e-03, -1.14150005e-03,
           2.68032242e-02])],
  'hand_type': <HandType.

In [6]:
ycb_optimize_dataset_list = []

for obj_idx in grasp_poses.keys():
    hand_pos, hand_euler, object_pose, hand_6drot = get_obj_centric_pose_for_opt(grasp_poses, obj_idx)
    robot_pose_joint = grasp_poses[obj_idx]['robot_pose'][0]
    map_idx = [0, 5, 10, 15, 18, 1, 6, 11, 16, 2, 7, 12, 17, 3, 8, 13, 4, 9, 14]
    mapped_joint = [robot_pose_joint[6:][i] for i in map_idx]

    robot_joint_dict = defaultdict(float)
    for i, joint_name in enumerate(joint_names):
        robot_joint_dict[joint_name] = mapped_joint[i]
    for i, name in enumerate(translation_names):
        robot_joint_dict[name] = hand_pos[i]
    for i, name in enumerate(rot_names):
        robot_joint_dict[name] = hand_euler[i]

    ycb_optimize_dataset = defaultdict(dict)
    ycb_optimize_dataset['qpos'] = robot_joint_dict
    ycb_optimize_dataset['object_code'] = grasp_poses[obj_idx]['target_object_name']
    ycb_optimize_dataset['object_pose'] = object_pose
    ycb_optimize_dataset['hand_rot6d'] = hand_6drot
    ycb_optimize_dataset['idx'] = obj_idx

    ycb_optimize_dataset_list.append(ycb_optimize_dataset)

ycb_optimize_dataset_list

[[ 0.48774798 -0.6438398  -0.58955256  0.08526855]
 [ 0.48101253  0.76177328 -0.43396821  0.01012099]
 [ 0.7285114  -0.07191505  0.68124839 -0.13418037]
 [ 0.          0.          0.          1.        ]]
Object-Centric Hand Euler (XYZ, rad): [ 0.56719559 -0.63050478  0.92247366]
Object-Centric Hand Position: [ 0.08526855  0.01012099 -0.13418037]
[[ 0.37692867 -0.92595786 -0.02295239  0.02315089]
 [-0.09274996 -0.01307688 -0.99560356  0.17383158]
 [ 0.92158679  0.37740036 -0.09081161 -0.02917738]
 [ 0.          0.          0.          1.        ]]
Object-Centric Hand Euler (XYZ, rad): [ 1.66175724 -0.02295441  1.18421094]
Object-Centric Hand Position: [ 0.02315089  0.17383158 -0.02917738]
[[ 0.08171717  0.90403577  0.41957316 -0.09206855]
 [ 0.93371386  0.07779156 -0.34946659 -0.02225276]
 [-0.34856955  0.4203187  -0.83775382  0.09974146]
 [ 0.          0.          0.          1.        ]]
Object-Centric Hand Euler (XYZ, rad): [ 2.74639222  0.43297504 -1.48064979]
Object-Centric Hand P

[defaultdict(dict,
             {'qpos': defaultdict(float,
                          {'thumb_cmc_roll': -0.19065852463245392,
                           'thumb_cmc_yaw': 0.19434407353401184,
                           'thumb_cmc_pitch': 0.10742746293544769,
                           'thumb_mcp': 0.033508218824863434,
                           'thumb_ip': 0.026803224238008263,
                           'index_mcp_roll': 0.26260000467300415,
                           'index_mcp_pitch': -0.0010000000474974513,
                           'index_pip': -0.0010000000474974513,
                           'index_dip': -0.0011162000530166552,
                           'middle_mcp_roll': 0.10580864548683167,
                           'middle_mcp_pitch': -0.0010000000474974513,
                           'middle_pip': -0.0010000000474974513,
                           'middle_dip': -0.001141500054218341,
                           'ring_mcp_pitch': 0.3966025114059448,
                      

In [7]:
ycb_optimize_dataset_list

[defaultdict(dict,
             {'qpos': defaultdict(float,
                          {'thumb_cmc_roll': -0.19065852463245392,
                           'thumb_cmc_yaw': 0.19434407353401184,
                           'thumb_cmc_pitch': 0.10742746293544769,
                           'thumb_mcp': 0.033508218824863434,
                           'thumb_ip': 0.026803224238008263,
                           'index_mcp_roll': 0.26260000467300415,
                           'index_mcp_pitch': -0.0010000000474974513,
                           'index_pip': -0.0010000000474974513,
                           'index_dip': -0.0011162000530166552,
                           'middle_mcp_roll': 0.10580864548683167,
                           'middle_mcp_pitch': -0.0010000000474974513,
                           'middle_pip': -0.0010000000474974513,
                           'middle_dip': -0.001141500054218341,
                           'ring_mcp_pitch': 0.3966025114059448,
                      

In [8]:
# store dict in a specified path as npy file
import os
import json
output_path = '/home/guizhewei/guizhewei/grasp_pose_dataset/unoptimized/dexycb_robot_joint_dict_1021_0045_omnihand.npy'
if not os.path.exists(os.path.dirname(output_path)):
    os.makedirs(os.path.dirname(output_path))
np.save(output_path, ycb_optimize_dataset_list, allow_pickle=True)